# CS781 Phase 4 — Court Judgment Prediction (CJPE)
All experiments start from the Phase 1 LegalLM (decoder-only, 12L / 768d, ctx 1024).

| Submission | Technique | Long-doc representation |
|---|---|---|
| 01_fullft_hier | Full fine-tuning (also the KD teacher) | head + tail chunks -> hierarchical attention pooling |
| 02_lora_hier | LoRA on qkv/proj/w12/w3 (compulsory) | same |
| 03_adapter_hier | Bottleneck adapters (Houlsby style) | same |
| 04_kd_student6 | Knowledge distillation: 12L teacher -> 6L student | same |
| 05_partial_top4 | Partial / layer-wise FT (top 4 blocks only) | same |
| 06_ensemble_avg | Probability average of all models above | — |
| 07_lora_tail1024 | LoRA, flat truncation baseline | last 1024 tokens only |

Every run writes `/kaggle/working/<name>.zip` (containing `prediction.json`) plus dev-side
files for `evaluate_phase4.py`, and a row in `results.csv` (F1, P, R, params, memory, time).

In [ ]:
# Config
import os, sys, json, math, time, random, re, shutil, zipfile, gc, csv
from contextlib import nullcontext
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.checkpoint
from sklearn.metrics import f1_score, precision_score, recall_score

SMOKE = os.environ.get("CJPE_SMOKE", "0") == "1"   # set True for a tiny dry run (a few docs, ~1 min/experiment)
RUN_START = time.time()

INPUT_ROOT = os.environ.get("CJPE_INPUT", "/kaggle/input")
OUT = os.environ.get("CJPE_OUT", "/kaggle/working")
os.makedirs(OUT, exist_ok=True)

CFG = dict(
    seed=42,
    chunk_len=512,        # tokens per chunk
    overlap=64,           # overlap between consecutive chunks
    max_chunks=6,         # chunks kept per document
    head_chunks=1,        # of which taken from the start; the rest come from the END of the judgment
    micro_docs=2,         # documents per forward pass while training
    accum=8,              # gradient accumulation -> effective batch = 16 documents
    infer_docs=8,         # documents per forward pass at inference
    kd_pool=8000,         # max training docs the KD student uses (teacher logits computed for these)
    n_evals=3,            # dev evaluations per run (best checkpoint kept)
    total_hours=4.6,      # hard wall-clock guard for the whole notebook
    holdout_frac=0.2,     # only used when train.json is missing (dev is split into train/val)
)

# minutes = training-time budget per experiment; the run stops at min(epochs, budget)
EXPS = [
    dict(name="01_fullft_hier",   method="full",    lr=3e-5, head_lr=1e-3, epochs=3, minutes=45),
    dict(name="02_lora_hier",     method="lora",    lr=4e-4, head_lr=1e-3, epochs=5, minutes=35, r=16, alpha=32),
    dict(name="03_adapter_hier",  method="adapter", lr=1e-3, head_lr=1e-3, epochs=5, minutes=35, bottleneck=64),
    dict(name="04_kd_student6",   method="kd",      lr=6e-5, head_lr=1e-3, epochs=5, minutes=28,
         layers=[0, 2, 4, 7, 9, 11], T=2.0, alpha=0.7),
    dict(name="05_partial_top4",  method="partial", lr=6e-5, head_lr=1e-3, epochs=5, minutes=22, train_top=4),
    dict(name="07_lora_tail1024", method="lora",    lr=4e-4, head_lr=1e-3, epochs=5, minutes=18, r=16, alpha=32,
         chunk_len=1024, max_chunks=1, head_chunks=0, micro_docs=4, accum=4, optional=True),
]

if SMOKE:
    CFG.update(chunk_len=64, overlap=8, max_chunks=3, micro_docs=2, accum=2, infer_docs=4,
               kd_pool=16, n_evals=2, total_hours=1.0)
    for e in EXPS:
        e.update(epochs=1, minutes=0.3)
        if e["name"].startswith("07"):
            e.update(chunk_len=128, micro_docs=2, accum=2)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = device.type == "cuda"
print("device:", device, torch.cuda.get_device_name(0) if USE_AMP else "")


def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)


seed_all(CFG["seed"])


def elapsed_h():
    return (time.time() - RUN_START) / 3600

In [ ]:
# Locate inputs (searches every attached dataset)
def find_files(name):
    hits = []
    for root, _, files in os.walk(INPUT_ROOT):
        if name in files:
            hits.append(os.path.join(root, name))
    return sorted(hits)


def load_split(name, need_label):
    for p in find_files(name):
        try:
            d = json.load(open(p, encoding="utf-8"))
        except Exception:
            continue
        if isinstance(d, list) and d and isinstance(d[0], dict) and "text" in d[0] and "id" in d[0] \
                and (not need_label or "label" in d[0]):
            print(f"  {name}: {p} ({len(d)} rows)")
            return d
    return None


MODEL_BIN = find_files("pytorch_model.bin")[0]
MODEL_DIR = os.path.dirname(MODEL_BIN)
print("model dir:", MODEL_DIR)

print("data:")
train_raw = load_split("train.json", need_label=True)
dev_raw = load_split("dev.json", need_label=True)
test_raw = load_split("test.json", need_label=False)
assert dev_raw is not None and test_raw is not None, "dev.json / test.json not found under /kaggle/input"

In [ ]:
# Load the Phase 1 model code + tokenizer
shutil.copy(os.path.join(MODEL_DIR, "model.py"), os.path.join(OUT, "legal_lm_model.py"))
sys.path.insert(0, OUT)
from legal_lm_model import LegalLM, ModelConfig  # noqa: E402
from tokenizers import Tokenizer  # noqa: E402

LM_CFG = json.load(open(os.path.join(MODEL_DIR, "config.json")))
BASE_SD = torch.load(MODEL_BIN, map_location="cpu")
tokenizer = Tokenizer.from_file(os.path.join(MODEL_DIR, "tokenizer.json"))
PAD_ID = 0  # <|endoftext|>; padding goes on the right, so causal attention never sees it
print("LM config:", LM_CFG)

In [ ]:
# Data: dedupe, split, tokenize
def dedupe(rows, tag):
    seen, out = set(), []
    for r in rows:
        if r["id"] not in seen:
            seen.add(r["id"]); out.append(r)
    if len(out) != len(rows):
        print(f"  {tag}: removed {len(rows) - len(out)} duplicate IDs -> {len(out)}")
    return out


dev_rows = dedupe(dev_raw, "dev")
test_rows = dedupe(test_raw, "test")
assert len(test_rows) == len(test_raw), "test.json has duplicate IDs"
for r in test_rows:          # test labels must never be used
    r.pop("label", None)

if train_raw is not None:
    train_rows = dedupe(train_raw, "train")
    dev_ids = {r["id"] for r in dev_rows}
    n0 = len(train_rows)
    train_rows = [r for r in train_rows if r["id"] not in dev_ids]
    if n0 != len(train_rows):
        print(f"  removed {n0 - len(train_rows)} train docs that also appear in dev")
    val_rows = dev_rows
    MODE = "train->dev"
else:
    print("\n" + "!" * 80 + "\n  train.json NOT FOUND - training on 80% of (deduplicated) dev, validating on 20%.\n"
          "  Add the Phase_4_CJPE train.json as an input for much better results.\n" + "!" * 80)
    rng = random.Random(CFG["seed"])
    by_lab = {0: [], 1: []}
    for r in dev_rows:
        by_lab[int(r["label"])].append(r)
    train_rows, val_rows = [], []
    for lab, rs in by_lab.items():
        rng.shuffle(rs)
        k = int(round(len(rs) * CFG["holdout_frac"]))
        val_rows += rs[:k]; train_rows += rs[k:]
    MODE = "dev-split"

if SMOKE:
    rng = random.Random(0)
    train_rows = rng.sample(train_rows, 24); val_rows = rng.sample(val_rows, 12); test_rows = test_rows[:10]

print(f"MODE={MODE} | train {len(train_rows)} | val {len(val_rows)} | test {len(test_rows)}")


def normalize(t):
    return re.sub(r"\s+", " ", t).strip()


def tokenize_rows(rows, tag):
    t0 = time.time(); docs = []
    for i in range(0, len(rows), 500):
        batch = rows[i:i + 500]
        enc = tokenizer.encode_batch([normalize(r["text"]) for r in batch])
        for r, e in zip(batch, enc):
            ids = np.asarray(e.ids if e.ids else [PAD_ID], dtype=np.uint16)
            docs.append(dict(id=r["id"], label=int(r["label"]) if "label" in r else -1, ids=ids, ntok=len(ids)))
    lens = np.array([d["ntok"] for d in docs])
    print(f"  {tag}: {len(docs)} docs | tokens p50 {int(np.median(lens))} p90 {int(np.percentile(lens, 90))} "
          f"max {lens.max()} | {time.time() - t0:.0f}s")
    return docs


train_docs = tokenize_rows(train_rows, "train")
val_docs = tokenize_rows(val_rows, "val")
test_docs = tokenize_rows(test_rows, "test")
del train_raw, dev_raw, test_raw, train_rows, val_rows, test_rows, dev_rows
gc.collect()

y_train = np.array([d["label"] for d in train_docs])
cnt = np.bincount(y_train, minlength=2)
CLASS_W = torch.tensor(len(y_train) / (2.0 * np.maximum(cnt, 1)), dtype=torch.float32)
print("train label counts:", cnt.tolist(), "| class weights:", CLASS_W.tolist())

json.dump([{"id": d["id"], "gold": d["label"]} for d in val_docs],
          open(os.path.join(OUT, "val_truth.json"), "w"))

In [ ]:
# Chunking + batching
def make_chunks(ids, L, overlap, K, h):
    """Sliding windows over the whole doc; if more than K, keep the first h and the last K-h."""
    n = len(ids)
    if n <= L:
        return [ids]
    stride = L - overlap
    starts = list(range(0, n - L + 1, stride))
    if starts[-1] != n - L:
        starts.append(n - L)
    if len(starts) > K:
        starts = starts[:h] + (starts[-(K - h):] if K - h > 0 else [])
    return [ids[s:s + L] for s in starts]


def collate(docs, cc, keep="all"):
    """keep: 'all' | 'head' (first chunk only) | 'tail' (drop the first chunk) — for section ablation."""
    seqs, doc_idx, slot, pos_end, first = [], [], [], [], []
    for b, d in enumerate(docs):
        ch = make_chunks(d["ids"], cc["chunk_len"], cc["overlap"], cc["max_chunks"], cc["head_chunks"])
        m, s = len(ch), 0
        for i, c in enumerate(ch):
            if m > 1 and ((keep == "head" and i > 0) or (keep == "tail" and i == 0)):
                continue
            seqs.append(c); doc_idx.append(b); slot.append(s); pos_end.append(m - 1 - i); first.append(int(i == 0))
            s += 1
    T = max(len(c) for c in seqs)
    ids = torch.full((len(seqs), T), PAD_ID, dtype=torch.long)
    lens = torch.empty(len(seqs), dtype=torch.long)
    for j, c in enumerate(seqs):
        ids[j, :len(c)] = torch.from_numpy(c.astype(np.int64)); lens[j] = len(c)
    return dict(ids=ids, lens=lens, doc_idx=torch.tensor(doc_idx), slot=torch.tensor(slot),
                pos_end=torch.tensor(pos_end), first=torch.tensor(first), n_docs=len(docs),
                labels=torch.tensor([d["label"] for d in docs]))


def to_dev(batch):
    return {k: (v.to(device, non_blocking=True) if torch.is_tensor(v) else v) for k, v in batch.items()}

In [ ]:
# Model: Phase 1 backbone + hierarchical chunk->document classifier
class CJPEModel(nn.Module):
    def __init__(self, lm, d_doc=256, max_slots=64, dropout=0.1):
        super().__init__()
        self.lm = lm
        d = lm.cfg.d_model
        self.frozen_below = 0      # blocks below this index run under no_grad (partial FT)
        self.grad_ckpt = False
        # chunk vector = [mean-pool ; last-token] of the final hidden states
        self.chunk_proj = nn.Sequential(nn.Linear(2 * d, d_doc), nn.GELU(), nn.LayerNorm(d_doc))
        self.pos_end = nn.Embedding(max_slots, d_doc)     # position counted from the END of the judgment
        self.first_flag = nn.Embedding(2, d_doc)          # is this the opening chunk?
        self.ln1 = nn.LayerNorm(d_doc)
        self.mha = nn.MultiheadAttention(d_doc, 4, dropout=dropout, batch_first=True)
        self.ln2 = nn.LayerNorm(d_doc)
        self.ff = nn.Sequential(nn.Linear(d_doc, 2 * d_doc), nn.GELU(), nn.Dropout(dropout), nn.Linear(2 * d_doc, d_doc))
        self.att_q = nn.Linear(d_doc, 1)
        self.drop = nn.Dropout(dropout)
        self.cls = nn.Linear(d_doc, 2)
        self.max_slots = max_slots

    def head_parameters(self):
        return [p for n, p in self.named_parameters() if not n.startswith("lm.")]

    def encode(self, ids, lens):
        lm, T = self.lm, ids.shape[1]
        cos, sin = lm.rope_cos[:T][None, None], lm.rope_sin[:T][None, None]
        fb = self.frozen_below
        with (torch.no_grad() if fb > 0 else nullcontext()):
            x = lm.tok(ids)
            for blk in lm.blocks[:fb]:
                x = blk(x, cos, sin)
        if fb > 0:
            x = x.detach()
        for blk in lm.blocks[fb:]:
            if self.grad_ckpt and self.training:
                x = torch.utils.checkpoint.checkpoint(blk, x, cos, sin, use_reentrant=False)
            else:
                x = blk(x, cos, sin)
        x = lm.norm(x).float()
        mask = (torch.arange(T, device=ids.device)[None] < lens[:, None]).float()
        mean = (x * mask[..., None]).sum(1) / lens[:, None].float()
        last = x[torch.arange(x.shape[0], device=ids.device), lens - 1]
        return torch.cat([mean, last], -1)

    def forward(self, b):
        v = self.chunk_proj(self.encode(b["ids"], b["lens"]))                       # [C, d_doc]
        v = v + self.pos_end(b["pos_end"].clamp(max=self.max_slots - 1)) + self.first_flag(b["first"])
        B, S = b["n_docs"], int(b["slot"].max()) + 1
        H = v.new_zeros(B, S, v.shape[-1])
        H[b["doc_idx"], b["slot"]] = v
        mask = torch.zeros(B, S, dtype=torch.bool, device=v.device)
        mask[b["doc_idx"], b["slot"]] = True
        h = self.ln1(H)
        H = H + self.mha(h, h, h, key_padding_mask=~mask, need_weights=False)[0]
        H = H + self.ff(self.ln2(H))
        w = self.att_q(H).squeeze(-1).float().masked_fill(~mask, float("-inf")).softmax(-1)
        doc = (w[..., None] * H.float()).sum(1)
        return self.cls(self.drop(doc)).float(), w

In [ ]:
# Adaptation methods: LoRA, adapters, freezing, student construction
class LoRALinear(nn.Module):
    def __init__(self, base: nn.Linear, r, alpha, dropout=0.05):
        super().__init__()
        self.base = base
        self.A = nn.Parameter(torch.empty(r, base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.scale = alpha / r
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        return self.base(x) + (self.drop(x) @ self.A.t().to(x.dtype)) @ self.B.t().to(x.dtype) * self.scale


class Adapter(nn.Module):
    def __init__(self, d, b):
        super().__init__()
        self.down, self.up = nn.Linear(d, b), nn.Linear(b, d)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)

    def forward(self, h):
        return h + self.up(F.gelu(self.down(h)))


class WithAdapter(nn.Module):
    """Wraps a sub-layer (attention or MLP) and adapts its output before the residual add."""
    def __init__(self, sub, d, b):
        super().__init__()
        self.sub, self.adapter = sub, Adapter(d, b)

    def forward(self, *a, **k):
        return self.adapter(self.sub(*a, **k))


def fresh_lm(n_layer=None, layers=None):
    cfg = dict(LM_CFG)
    if layers is not None:
        cfg["n_layer"] = len(layers)
    lm = LegalLM(ModelConfig(**cfg))
    if layers is None:
        lm.load_state_dict(BASE_SD, strict=True)
    else:  # student: copy embeddings / final norm and a subset of blocks
        sd = {}
        for k, v in BASE_SD.items():
            if k.startswith("blocks."):
                i = int(k.split(".")[1])
                if i in layers:
                    sd[k.replace(f"blocks.{i}.", f"blocks.{layers.index(i)}.", 1)] = v
            else:
                sd[k] = v
        lm.load_state_dict(sd, strict=True)
    return lm


def build_model(exp):
    m = exp["method"]
    if m == "kd":
        model = CJPEModel(fresh_lm(layers=exp["layers"]))
    else:
        model = CJPEModel(fresh_lm())
    lm = model.lm
    if m in ("full", "kd"):
        pass
    elif m == "partial":
        k = len(lm.blocks) - exp["train_top"]
        for p in lm.parameters():
            p.requires_grad = False
        for blk in lm.blocks[k:]:
            for p in blk.parameters():
                p.requires_grad = True
        for p in lm.norm.parameters():
            p.requires_grad = True
        model.frozen_below = k
    elif m == "lora":
        for p in lm.parameters():
            p.requires_grad = False
        for blk in lm.blocks:
            blk.attn.qkv = LoRALinear(blk.attn.qkv, exp["r"], exp["alpha"])
            blk.attn.proj = LoRALinear(blk.attn.proj, exp["r"], exp["alpha"])
            blk.mlp.w12 = LoRALinear(blk.mlp.w12, exp["r"], exp["alpha"])
            blk.mlp.w3 = LoRALinear(blk.mlp.w3, exp["r"], exp["alpha"])
    elif m == "adapter":
        for p in lm.parameters():
            p.requires_grad = False
        d = lm.cfg.d_model
        for blk in lm.blocks:
            blk.attn = WithAdapter(blk.attn, d, exp["bottleneck"])
            blk.mlp = WithAdapter(blk.mlp, d, exp["bottleneck"])
            for p in list(blk.norm1.parameters()) + list(blk.norm2.parameters()):
                p.requires_grad = True
        for p in lm.norm.parameters():
            p.requires_grad = True
    else:
        raise ValueError(m)
    return model.to(device)


def param_stats(model):
    seen, total, train = set(), 0, 0
    for p in model.parameters():
        if id(p) in seen:
            continue
        seen.add(id(p)); total += p.numel(); train += p.numel() if p.requires_grad else 0
    return total, train

In [ ]:
# Training / inference / metrics
def macro_f1(y, p):
    return f1_score(y, p, average="macro", labels=[0, 1], zero_division=0)


def tune_threshold(y, prob):
    best_t, best_f = 0.5, macro_f1(y, (prob >= 0.5).astype(int))
    for t in np.arange(0.25, 0.751, 0.01):
        f = macro_f1(y, (prob >= t).astype(int))
        if f > best_f + 1e-9:
            best_t, best_f = float(t), f
    return best_t, best_f


@torch.no_grad()
def predict(model, docs, cc, keep="all"):
    model.eval()
    probs, logits_all, first_w, t0 = [], [], [], time.time()
    order = np.argsort([d["ntok"] for d in docs])          # length-sorted batches = less padding
    for i in range(0, len(docs), cc["infer_docs"]):
        idx = order[i:i + cc["infer_docs"]]
        b = to_dev(collate([docs[j] for j in idx], cc, keep))
        with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
            logits, w = model(b)
        logits_all.append((idx, logits.float().cpu()))
        fw = w[:, 0].cpu().numpy()                            # weight on slot 0 (= opening chunk when keep='all')
        nslots = (w > 0).sum(1).cpu().numpy()
        first_w += [(fw[k], nslots[k]) for k in range(len(idx))]
    if USE_AMP:
        torch.cuda.synchronize()
    out = torch.zeros(len(docs), 2)
    for idx, lg in logits_all:
        out[torch.as_tensor(idx)] = lg
    probs = out.softmax(-1)[:, 1].numpy()
    ms_per_doc = 1000 * (time.time() - t0) / max(1, len(docs))
    return probs, out.numpy(), ms_per_doc, first_w


def evaluate(model, docs, cc):
    prob, _, _, _ = predict(model, docs, cc)
    y = np.array([d["label"] for d in docs])
    return macro_f1(y, (prob >= 0.5).astype(int)), tune_threshold(y, prob)[1], prob


def train_model(model, exp, cc, docs, kd_logits=None):
    head_ids = {id(p) for p in model.head_parameters()}
    lm_params = [p for p in model.parameters() if p.requires_grad and id(p) not in head_ids]
    head_params = [p for p in model.head_parameters() if p.requires_grad]
    groups = []
    for ps, lr in ((lm_params, exp["lr"]), (head_params, exp["head_lr"])):
        dec = [p for p in ps if p.dim() >= 2]
        nod = [p for p in ps if p.dim() < 2]
        groups += [dict(params=dec, lr=lr, weight_decay=0.01), dict(params=nod, lr=lr, weight_decay=0.0)]
    groups = [g for g in groups if g["params"]]
    for g in groups:
        g["base_lr"] = g["lr"]
    opt = torch.optim.AdamW(groups, betas=(0.9, 0.98), eps=1e-6)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    cw = CLASS_W.to(device)

    n, mb, acc = len(docs), cc["micro_docs"], cc["accum"]
    spe = math.ceil(n / (mb * acc))
    sched = dict(total=exp["epochs"] * spe)
    warm = max(3, int(0.06 * sched["total"]))
    budget = exp["minutes"] * 60
    eval_at, t_start, step, best = set(), time.time(), 0, (-1.0, None, None)
    labels_all = torch.tensor([d["label"] for d in docs])
    kd_T, kd_a = exp.get("T", 2.0), exp.get("alpha", 0.7)

    def lr_mult(s):
        if s < warm:
            return (s + 1) / warm
        return max(0.05, (sched["total"] - s) / max(1, sched["total"] - warm))

    def snapshot():
        return {k: v.detach().to("cpu", copy=True) for k, v in model.state_dict().items()}

    model.train()
    done = False
    t_measure = None
    for ep in range(exp["epochs"]):
        perm = np.random.permutation(n)
        micro = [perm[i:i + mb] for i in range(0, n, mb)]
        for mi in range(0, len(micro), acc):
            group = micro[mi:mi + acc]
            for g in opt.param_groups:
                g["lr"] = g["base_lr"] * lr_mult(step)
            loss_v = 0.0
            for idx in group:
                b = to_dev(collate([docs[j] for j in idx], cc))
                try:
                    with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
                        logits, _ = model(b)
                    y = b["labels"]
                    loss = F.cross_entropy(logits, y, weight=cw)
                    if kd_logits is not None:
                        t_log = kd_logits[torch.as_tensor(idx)].to(device)
                        kl = F.kl_div(F.log_softmax(logits / kd_T, -1), F.softmax(t_log / kd_T, -1),
                                      reduction="batchmean") * kd_T ** 2
                        loss = kd_a * kl + (1 - kd_a) * loss
                    scaler.scale(loss / len(group)).backward()
                    loss_v += loss.item() / len(group)
                except torch.cuda.OutOfMemoryError:
                    print("  OOM -> enabling gradient checkpointing, skipping this micro-batch")
                    opt.zero_grad(set_to_none=True); torch.cuda.empty_cache(); model.grad_ckpt = True
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_([p for g in groups for p in g["params"]], 1.0)
            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
            step += 1

            if step == 2:
                t_measure = time.time()
            if step == 7:                                 # measure speed, fit the schedule into the budget
                sec_per_step = (time.time() - t_measure) / 5
                used = time.time() - t_start
                fit = step + int(max(0.0, budget - used) / sec_per_step)
                sched["total"] = max(step + 1, min(sched["total"], fit))
                eval_at = {max(step + 1, int(round(sched["total"] * f)))
                           for f in np.linspace(0.5, 1.0, CFG["n_evals"])}
                print(f"  {sec_per_step:.2f}s/step | planned {exp['epochs'] * spe} steps -> running {sched['total']} "
                      f"({sched['total'] * mb * acc / n:.2f} epochs) | evals at {sorted(eval_at)}")
            if step % 20 == 0:
                print(f"  step {step}/{sched['total']} | loss {loss_v:.4f} | lr x{lr_mult(step):.2f} | "
                      f"{(time.time() - t_start) / 60:.1f} min")
            last = step >= sched["total"] or (time.time() - t_start) > budget * 1.15 \
                or elapsed_h() > CFG["total_hours"] - 0.15
            if step in eval_at or last:
                f05, ftuned, _ = evaluate(model, val_docs, cc)
                print(f"  [eval step {step}] val macro-F1 @0.5 {f05:.4f} | tuned {ftuned:.4f}")
                if f05 > best[0]:
                    best = (f05, step, snapshot())
                model.train()
            if last:
                done = True
                break
        if done:
            break
    if best[2] is None:          # very short runs that never hit an eval point
        best = (evaluate(model, val_docs, cc)[0], step, snapshot())
    model.load_state_dict(best[2])
    print(f"  best val macro-F1 @0.5 = {best[0]:.4f} at step {best[1]} | train {(time.time() - t_start) / 60:.1f} min")
    return dict(train_min=(time.time() - t_start) / 60, steps=step, best_step=best[1])

In [ ]:
# Output helpers
RESULTS = []
VAL_PROBS, TEST_PROBS = {}, {}
y_val = np.array([d["label"] for d in val_docs])


def write_submission(name, test_prob, thr):
    preds = [{"id": d["id"], "pred": int(p >= thr)} for d, p in zip(test_docs, test_prob)]
    assert len(preds) == len(test_docs) and len({p["id"] for p in preds}) == len(preds)
    tag = ("smoke_" if SMOKE else "") + name
    jpath = os.path.join(OUT, f"{tag}_prediction.json")
    json.dump(preds, open(jpath, "w"))
    with zipfile.ZipFile(os.path.join(OUT, f"{tag}.zip"), "w", zipfile.ZIP_DEFLATED) as z:
        z.write(jpath, arcname="prediction.json")
    print(f"  -> {tag}.zip  (test pred=1 rate {np.mean([p['pred'] for p in preds]):.3f})")


def record(name, method, val_prob, test_prob, extra):
    thr, f_t = tune_threshold(y_val, val_prob)
    pv = (val_prob >= thr).astype(int)
    row = dict(name=name, method=method, val_macro_f1_05=round(macro_f1(y_val, (val_prob >= 0.5).astype(int)), 4),
               val_macro_f1_tuned=round(f_t, 4), threshold=round(thr, 2),
               val_precision=round(precision_score(y_val, pv, zero_division=0), 4),
               val_recall=round(recall_score(y_val, pv, zero_division=0), 4), **extra)
    RESULTS.append(row)
    VAL_PROBS[name], TEST_PROBS[name] = val_prob, test_prob
    json.dump([{"id": d["id"], "pred": int(p)} for d, p in zip(val_docs, pv)],
              open(os.path.join(OUT, f"val_pred_{name}.json"), "w"))
    with open(os.path.join(OUT, f"val_details_{name}.csv"), "w", newline="") as f:
        w = csv.writer(f); w.writerow(["id", "gold", "prob1", "pred", "n_tokens", "error"])
        for d, p, q in zip(val_docs, val_prob, pv):
            err = "" if q == d["label"] else ("FP" if q == 1 else "FN")
            w.writerow([d["id"], d["label"], f"{p:.4f}", int(q), d["ntok"], err])
    write_submission(name, test_prob, thr)
    with open(os.path.join(OUT, "results.csv"), "w", newline="") as f:
        keys = list(dict.fromkeys(k for r in RESULTS for k in r))
        w = csv.DictWriter(f, fieldnames=keys); w.writeheader(); w.writerows(RESULTS)
    print("  ", row)


def exp_cc(exp):
    return {**CFG, **{k: exp[k] for k in ("chunk_len", "overlap", "max_chunks", "head_chunks", "micro_docs",
                                          "accum", "infer_docs") if k in exp}}


def run_experiment(exp, kd_logits=None, train_subset=None):
    print("\n" + "=" * 90 + f"\n{exp['name']}  ({exp['method']})  | notebook time {elapsed_h():.2f} h\n" + "=" * 90)
    seed_all(CFG["seed"])
    cc = exp_cc(exp)
    model = build_model(exp)
    total, trainable = param_stats(model)
    print(f"  params: total {total / 1e6:.2f}M | trainable {trainable / 1e6:.3f}M ({100 * trainable / total:.2f}%)")
    if USE_AMP:
        torch.cuda.reset_peak_memory_stats()
    docs = train_docs if train_subset is None else [train_docs[i] for i in train_subset]
    info = train_model(model, exp, cc, docs, kd_logits)
    peak = torch.cuda.max_memory_allocated() / 2 ** 30 if USE_AMP else 0.0
    val_prob, _, ms_val, first_w = predict(model, val_docs, cc)
    test_prob, _, ms_test, _ = predict(model, test_docs, cc)
    multi = [fw for fw, ns in first_w if ns > 1]
    extra = dict(total_params_M=round(total / 1e6, 2), trainable_params_M=round(trainable / 1e6, 3),
                 trainable_pct=round(100 * trainable / total, 2), size_fp16_MB=round(total * 2 / 2 ** 20, 1),
                 peak_train_mem_GB=round(peak, 2), train_min=round(info["train_min"], 1),
                 steps=info["steps"], infer_ms_per_doc=round((ms_val + ms_test) / 2, 1),
                 chunks=f"{cc['max_chunks']}x{cc['chunk_len']}",
                 attn_on_opening_chunk=round(float(np.mean(multi)), 3) if multi else None)
    record(exp["name"], exp["method"], val_prob, test_prob, extra)
    return model, cc

In [ ]:
# 01 — Full fine-tuning (teacher)  + section ablation + teacher logits for KD
E = {e["name"]: e for e in EXPS}
teacher, tcc = run_experiment(E["01_fullft_hier"])

ab_rows = []
for keep in ("all", "head", "tail"):
    p, _, _, _ = predict(teacher, val_docs, tcc, keep=keep)
    ab_rows.append(dict(input=keep, val_macro_f1_05=round(macro_f1(y_val, (p >= 0.5).astype(int)), 4),
                        val_macro_f1_tuned=round(tune_threshold(y_val, p)[1], 4)))
    print("  section ablation:", ab_rows[-1])
with open(os.path.join(OUT, "section_ablation_teacher.csv"), "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(ab_rows[0])); w.writeheader(); w.writerows(ab_rows)

rng = np.random.default_rng(CFG["seed"])
KD_SUBSET = np.sort(rng.choice(len(train_docs), size=min(CFG["kd_pool"], len(train_docs)), replace=False))
t0 = time.time()
_, kd_logits_np, _, _ = predict(teacher, [train_docs[i] for i in KD_SUBSET], tcc)
KD_LOGITS = torch.tensor(kd_logits_np, dtype=torch.float32)
print(f"  teacher logits for {len(KD_SUBSET)} train docs in {(time.time() - t0) / 60:.1f} min")
del teacher
gc.collect(); torch.cuda.empty_cache() if USE_AMP else None

In [ ]:
# 02 — LoRA
m, _ = run_experiment(E["02_lora_hier"]); del m; gc.collect(); torch.cuda.empty_cache() if USE_AMP else None

In [ ]:
# 03 — Adapters
m, _ = run_experiment(E["03_adapter_hier"]); del m; gc.collect(); torch.cuda.empty_cache() if USE_AMP else None

In [ ]:
# 04 — Knowledge distillation (12-layer full-FT teacher -> 6-layer student initialised from Phase 1 layers)
m, _ = run_experiment(E["04_kd_student6"], kd_logits=KD_LOGITS, train_subset=KD_SUBSET)
del m; gc.collect(); torch.cuda.empty_cache() if USE_AMP else None

In [ ]:
# 05 — Partial / layer-wise fine-tuning (top 4 blocks)
m, _ = run_experiment(E["05_partial_top4"]); del m; gc.collect(); torch.cuda.empty_cache() if USE_AMP else None

In [ ]:
# 06 — Ensemble (probability average of every model trained so far)
names = [n for n in VAL_PROBS if not n.startswith("06")]
vp = np.mean([VAL_PROBS[n] for n in names], 0)
tp = np.mean([TEST_PROBS[n] for n in names], 0)
print("\n06_ensemble_avg over:", names)
record("06_ensemble_avg", "ensemble(" + "+".join(n[:2] for n in names) + ")", vp, tp, dict(steps=0))

In [ ]:
# 07 — Flat truncation baseline: LoRA on the last 1024 tokens only (skipped if out of time)
exp7 = E["07_lora_tail1024"]
if elapsed_h() + (exp7["minutes"] + 8) / 60 < CFG["total_hours"]:
    m, _ = run_experiment(exp7); del m; gc.collect(); torch.cuda.empty_cache() if USE_AMP else None
else:
    print(f"Skipping 07 — {elapsed_h():.2f} h used")

In [ ]:
# Summary
print(f"\nTotal notebook time: {elapsed_h():.2f} h | MODE={MODE}")
cols = ["name", "val_macro_f1_05", "val_macro_f1_tuned", "threshold", "val_precision", "val_recall",
        "total_params_M", "trainable_params_M", "peak_train_mem_GB", "train_min", "infer_ms_per_doc"]
print(" | ".join(cols))
for r in RESULTS:
    print(" | ".join(str(r.get(c, "")) for c in cols))
print("\nSubmission zips:", sorted(f for f in os.listdir(OUT) if f.endswith(".zip")))
print("Check any run locally:  python evaluate_phase4.py --truth val_truth.json --prediction val_pred_<name>.json")